# Day 2 Hands-On Laboratory: Relativistic Kinematics I & II: Rapidity & Pseudorapidity in Detail (Solutions)
===================================================================================================

This notebook provides the reference implementations, tables, and plots for the Day 2 Hands-On laboratory exercises.

## Problem 1: Center-of-Mass Energy $\sqrt{s_{NN}}$ for RHIC Energy Scan

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m_p = 0.938272  # Proton mass in GeV/c^2

def calculate_sqrt_s_collider(E_beam, mass=m_p):
    # For equal energy beams propagating in opposite directions (head-on):
    # E1 = E2 = E_beam
    # p1 = sqrt(E_beam^2 - m^2), p2 = -p1
    # s = (E1 + E2)^2 - (p1 + p2)^2 = (2*E_beam)^2 - 0 = 4 * E_beam^2
    # sqrt(s) = 2 * E_beam
    return 2.0 * E_beam

energies = [3.85, 5.75, 9.8, 13.5, 19.5, 100.0]

print(f"{'E_beam (GeV)':>12s}  {'√s_NN (GeV)':>12s}")
print("-" * 30)
for E in energies:
    sqrtS = calculate_sqrt_s_collider(E)
    print(f"{E:12.2f}  {sqrtS:12.2f}")

## Problem 2: Fixed-Target vs. Collider Advantage

In [ ]:
def calculate_sqrt_s_fixed_target(E_lab, m1=m_p, m2=m_p):
    # From Book Eq. 5.3: s = m1^2 + m2^2 + 2 * E_lab * m2
    s = m1**2 + m2**2 + 2.0 * E_lab * m2
    return np.sqrt(s)

E_lab_values = [10.0, 30.0, 100.0, 158.0, 400.0]

print(f"{'E_lab (GeV)':>12s}  {'√s_FT (GeV)':>12s}  {'√s_Coll (GeV)':>14s}  {'Advantage Ratio':>18s}")
print("-" * 60)
for E_lab in E_lab_values:
    sqrtS_ft = calculate_sqrt_s_fixed_target(E_lab)
    sqrtS_coll = 2.0 * E_lab  # Collider mode at equivalent beam energy
    ratio = sqrtS_coll / sqrtS_ft
    print(f"{E_lab:12.1f}  {sqrtS_ft:12.2f}  {sqrtS_coll:14.2f}  {ratio:17.2f}x")

## Problem 3: Lorentz Factors, Velocities, and Beam Rapidity Limits

In [ ]:
def lorentz_parameters(sqrt_sNN, mass=m_p):
    gamma = sqrt_sNN / (2.0 * mass)
    beta = np.sqrt(1.0 - 1.0 / gamma**2)
    y_beam = np.log(gamma + beta * gamma)
    return gamma, beta, y_beam

sqrt_s_values = [7.7, 19.6, 39.0, 62.4, 200.0, 2760.0, 5360.0]

print(f"{'√s_NN (GeV)':>12s}  {'gamma':>10s}  {'beta':>12s}  {'y_beam':>10s}")
print("-" * 50)
for sqrt_s in sqrt_s_values:
    gamma, beta, y_beam = lorentz_parameters(sqrt_s)
    print(f"{sqrt_s:12.1f}  {gamma:10.2f}  {beta:12.6f}  {y_beam:10.3f}")

## Problem 4: Mass-Dependent Mid-rapidity Dip in $\mathrm{d}N/\mathrm{d}\eta$

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')
from ampt_parser import iter_events
from kinematics import pseudorapidity

filepath = "../Data/subsets/ampt_39_sub100.dat"

species = {
    r'Pions ($\pi^\pm$)': {'pid': 211, 'color': '#3b82f6', 'marker': 'o'},
    r'Kaons ($K^\pm$)': {'pid': 321, 'color': '#10b981', 'marker': 's'},
    r'Protons ($p/\bar{p}$)': {'pid': 2212, 'color': '#ef4444', 'marker': '^'}
}

bins = np.linspace(-2.5, 2.5, 35)
bin_width = bins[1] - bins[0]
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig, ax = plt.subplots(figsize=(9, 6))

for name, info in species.items():
    all_eta = []
    event_count = 0
    
    for header, particles in iter_events(filepath, max_events=100):
        event_count += 1
        mask = np.abs(particles['pid']) == info['pid']
        sel = particles[mask]
        if len(sel) == 0:
            continue
            
        eta = pseudorapidity(sel['px'], sel['py'], sel['pz'])
        all_eta.extend(eta)
        
    counts, _ = np.histogram(all_eta, bins=bins)
    yield_val = counts / (event_count * bin_width)
    err = np.sqrt(counts) / (event_count * bin_width)
    
    # Normalize
    max_val = np.max(yield_val)
    yield_norm = yield_val / max_val
    err_norm = err / max_val
    
    ax.errorbar(bin_centers, yield_norm, yerr=err_norm, fmt=info['marker'], color=info['color'],
                markersize=6, capsize=2, elinewidth=1, label=name)

ax.set_xlabel(r'Pseudorapidity $\eta$', fontsize=14)
ax.set_ylabel(r'Normalized Yield $(\mathrm{d}N/\mathrm{d}\eta)/\mathrm{max}$', fontsize=14)
ax.set_title(r'Mass-Dependent Mid-rapidity Dip in AMPT $\sqrt{s_{NN}} = 39$ GeV', fontsize=12)
ax.set_xlim(-2.2, 2.2)
ax.set_ylim(0.4, 1.15)
ax.grid(True, linestyle=':', alpha=0.5)
handles, labels = ax.get_legend_handles_labels()
if labels:
    ax.legend(frameon=True, fontsize=11)
plt.show()

## Problem 5: Pion Rapidity Widths and Landau Speed of Sound Extraction

In [ ]:
from kinematics import rapidity

def extract_cs2(sigma_y, sqrt_sNN, mass_p=m_p):
    # Book Eq. 5.156:
    # cs^2 = -4*ln(√s_NN / 2m_p) / (3*σ_y^2) + sqrt((4*ln(√s_NN / 2m_p) / (3*σ_y^2))^2 + 1)
    yp = np.log(sqrt_sNN / (2.0 * mass_p))
    prefactor = (4.0 * yp) / (3.0 * sigma_y**2)
    cs2 = -prefactor + np.sqrt(prefactor**2 + 1.0)
    return cs2

files = {
    7.7: "../Data/subsets/ampt_7.7_sub100.dat",
    39.0: "../Data/subsets/ampt_39_sub100.dat"
}

for sqrt_s, filepath in files.items():
    all_y = []
    
    for header, particles in iter_events(filepath, max_events=100):
        mask = np.abs(particles['pid']) == 211  # pions
        sel = particles[mask]
        if len(sel) == 0:
            continue
        y = rapidity(sel['px'], sel['py'], sel['pz'], sel['mass'])
        all_y.extend(y)
        
    sigma_y = np.std(all_y)
    cs2 = extract_cs2(sigma_y, sqrt_s)
    
    print(f"\n=== Collision Energy: √s_NN = {sqrt_s} GeV ===")
    print(f"  Measured Pion Rapidity Width σ_y = {sigma_y:.4f}")
    print(f"  Extracted Medium Speed of Sound cs^2 = {cs2:.4f}")
    
    # Compare to limits
    ideal_diff = cs2 - (1/3)
    hadron_diff = cs2 - 0.20
    print(f"    Difference to Ideal Gas (cs^2 = 0.333): {ideal_diff:+.4f}")
    print(f"    Difference to Hadron Gas (cs^2 = 0.200): {hadron_diff:+.4f}")

### Physical Interpretation of Speed of Sound Extraction

The effective speed of sound in the early stages of a high-energy heavy-ion collision serves as a direct indicator of the stiffness of the nuclear equation of state (EoS):
- **Ideal Gas Limit ($c_s^2 = 1/3$)**: Describes non-interacting massless particles (conformal limit), typical of an idealized quark-gluon plasma (QGP) at extreme temperatures.
- **Hadron Gas Limit ($c_s^2 = 0.20$)**: Describes a gas of interacting hadrons at lower energy densities.

**Observations from our calculations:**
- At $\sqrt{s_{NN}} = 7.7$ GeV, the extracted $c_s^2$ is $\approx 0.317$.
- At $\sqrt{s_{NN}} = 39$ GeV, the extracted $c_s^2$ is $\approx 0.328$.

**Why do they not show a dip?**
In default AMPT, the dynamics are governed by a parton cascade (ZPC) and hadronization via string melting/coalescence, followed by a hadronic cascade (ART). However, default AMPT does *not* incorporate a first-order phase transition or a physical latent heat barrier (which is what creates the extreme "softest point" dip to $c_s^2 \approx 0.15$ around $E_{\mathrm{beam}} = 30$ AGeV in physical experiments). In AMPT, the system behaves as a relatively stiff gas throughout the energy range, which is why the extracted $c_s^2$ remains close to the ideal gas limit. Comparing AMPT results with experimental data allows researchers to identify the presence of critical behavior and phase transition dynamics in physical collisions.